
# Data Loading

We load the pre-processed training (2015–2018) and blind test (2019) datasets. These datasets have finished our feature engineering pipeline.

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
from pyspark.sql.functions import regexp_replace, mean, stddev, log1p, concat_ws
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

spark.conf.set("spark.sql.ansi.enabled", "false")

TEAM_DIR = "dbfs:/student-groups/Group_3_2"
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_test_df.parquet")

# Extract hour of day from scheduled departure time (not in checkpoint)
otpw_train_df = otpw_train_df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))
otpw_test_df = otpw_test_df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))

# Cast DISTANCE to numeric (stored as string in parquet)
otpw_train_df = otpw_train_df.withColumn("DISTANCE", col("DISTANCE").cast("double"))
otpw_test_df = otpw_test_df.withColumn("DISTANCE", col("DISTANCE").cast("double"))

print(f"Train (2015-2018): {otpw_train_df.count():,} rows x {len(otpw_train_df.columns)} columns")
print(f"Test  (2019):      {otpw_test_df.count():,} rows x {len(otpw_test_df.columns)} columns")

In [0]:
import os

# pandas requires /dbfs/ local path, not dbfs: URI
TEAM_DIR_LOCAL = TEAM_DIR.replace("dbfs:", "/dbfs")
CHECKPOINT_DATE = "041226"

def save_checkpoint(df, name):
    path = f"{TEAM_DIR_LOCAL}/{CHECKPOINT_DATE}_{name}.parquet"
    df.to_parquet(path)
    print(f"Checkpoint saved: {CHECKPOINT_DATE}_{name}")

def load_checkpoint(name):
    path = f"{TEAM_DIR_LOCAL}/{CHECKPOINT_DATE}_{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"Checkpoint loaded: {CHECKPOINT_DATE}_{name} ({len(df)} rows)")
        return df
    return None

In [0]:
label_col = "DEP_DEL15"

# Verify class balance
train_dist = otpw_train_df.groupBy(label_col).count().toPandas()
train_dist["pct"] = (train_dist["count"] / train_dist["count"].sum() * 100).round(2)
print(f"\nTrain class distribution:\n{train_dist}")

# Modeling Pipeline

We apply the modeling pipeline to the 4-year training dataset (2015 - 2018) with engineered features, evaluating Logistic Regression, Random Forest, Gradient Boosted Trees, and a Multilayer Perceptron. The 2019 dataset is held out as a blind test set that is never consulted during training or cross-validation.

### Build Spark ML Pipeline

The pipeline applies `StringIndexer`, `OneHotEncoder` for each categorical feature, then assembles all numeric and encoded categorical features into a single vector using `VectorAssembler`.

In [0]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Shared categorical features (used by LR and tree-based models)
categorical_features = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
    "VisibilityCat",
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

# 1. Logistic Regression Pipeline (Baseline)
#    Replaces delayed_flights_2_4hr_before with log_total_flights_2_4hr_before
#    to avoid multicollinearity (count = pct x total)
lr_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized",
    "2h_HourlyWindSpeed_log_stdized",
    "PrecipBinary",
    "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (log of total flights replaces raw count)
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

lr_assembler_inputs = lr_numeric_features + [f"{c}_ohe" for c in categorical_features]
lr_assembler = VectorAssembler(inputCols=lr_assembler_inputs, outputCol="features", handleInvalid="skip")
lr_pipeline_stages = indexers + encoders + [lr_assembler]

# 2. Tree-based Pipeline (RF, GBT)
tree_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized",
    "2h_HourlyWindSpeed_log_stdized",
    "PrecipBinary",
    "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (short-term airport congestion)
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

tree_assembler_inputs = tree_numeric_features + [f"{c}_ohe" for c in categorical_features]
tree_assembler = VectorAssembler(inputCols=tree_assembler_inputs, outputCol="features", handleInvalid="skip")
tree_pipeline_stages = indexers + encoders + [tree_assembler]

# 3. MLP Pipeline (reduced feature set for neural network)
#    Drops ORIGIN OHE (high cardinality might high dimentional sparse inputs for MLP), uses PageRank instead
mlp_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized", "2h_HourlyWindSpeed_log_stdized",
    "PrecipBinary", "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (replaces ORIGIN/DEST)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

mlp_categorical_features = [
    "OP_UNIQUE_CARRIER", "DAY_OF_WEEK", "MONTH", "DEP_HOUR", "VisibilityCat",
]

mlp_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_mlp_idx", handleInvalid="keep")
    for c in mlp_categorical_features
]
mlp_encoders = [
    OneHotEncoder(inputCol=f"{c}_mlp_idx", outputCol=f"{c}_mlp_ohe")
    for c in mlp_categorical_features
]
mlp_assembler_inputs = mlp_numeric_features + [f"{c}_mlp_ohe" for c in mlp_categorical_features]
mlp_assembler = VectorAssembler(inputCols=mlp_assembler_inputs, outputCol="features", handleInvalid="skip")
mlp_pipeline_stages = mlp_indexers + mlp_encoders + [mlp_assembler]

# Determine MLP input dimension from training data
mlp_prep = Pipeline(stages=mlp_pipeline_stages).fit(otpw_train_df)
sample_vec = mlp_prep.transform(otpw_train_df.limit(1)).select("features").head()[0]
mlp_input_dim = len(sample_vec)

# Summary
print(f"LR:    {len(lr_numeric_features)} numeric + {len(categorical_features)} categorical = {len(lr_assembler_inputs)} assembler inputs")
print(f"Tree:  {len(tree_numeric_features)} numeric + {len(categorical_features)} categorical = {len(tree_assembler_inputs)} assembler inputs")
print(f"MLP:   {len(mlp_numeric_features)} numeric + {len(mlp_categorical_features)} categorical = {mlp_input_dim} input dim (after OHE)")


### Evaluation Metrics

For airlines, the cost of a **false negative** (predicting on-time when the flight is actually delayed) is significantly higher than a **false positive** (predicting delayed when the flight departs on time):

- **False Negative (missed delay):** The airline fails to prepare: crew scheduling is disrupted last-minute, gate reassignments increases, passengers miss connections, and a single missed delay can trigger a chain of downstream issues. The financial and reputational cost is high.
- **False Positive (false alarm):** The airline pre-allocates standby resources that weren't needed: extra crew on standby, a backup gate reserved, passengers notified of a potential delay. The cost is moderate (some wasted resources) but overall harmless.

This motivates our choice of **F2-score** as the primary evaluation metric. F2 combines precision and recall into a single score but places twice the weight on recall, penalizing missed delays more heavily than false alarms. We use F2 rather than raw recall because optimizing recall alone is trivial (predict every flight as delayed), while F2 ensures the model maintains reasonable precision.

| Metric  | Role | Why |
|--------|------|-----|
| **F2**  | Primary | Balances precision and recall with 2x weight on recall, preventing missed delays while keeping predictions actionable |
| **Recall** | Supporting | Measures raw delay detection rate; F2's recall-weighted design ensures this stays high |
| **Precision** | Supporting | Ensures predicted delays are actionable, not just noise |
| **AUC-PR**  | Supporting | Captures the full precision-recall trade-off across all thresholds |

In [0]:
def evaluate_model(predictions, label_col="DEP_DEL15"):
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")

    tp = predictions.filter((col("prediction") == 1) & (col(label_col) == 1)).count()
    fp = predictions.filter((col("prediction") == 1) & (col(label_col) == 0)).count()
    fn = predictions.filter((col("prediction") == 0) & (col(label_col) == 1)).count()

    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f2 = round(5 * precision * recall / (4 * precision + recall), 4) if (precision + recall) > 0 else 0.0

    return {
        "AUC-PR": round(evaluator_pr.evaluate(predictions), 4),
        "Precision": precision,
        "Recall": recall,
        "F2": f2,
    }

### Modeling Approaches

We use **Logistic Regression** as the baseline. We then compare against tree-based models and a neural network:

- **Random Forest:** Handles non-linear feature interactions without explicit engineering. Robust to outliers and noisy features.
- **Gradient Boosted Trees:** Sequentially builds trees to correct previous errors, typically achieving higher predictive performance than Random Forest. More prone to overfitting.
- **Multilayer Perceptron (MLP):** A feed-forward neural network that learns non-linear decision boundaries through hidden layers. We use a reduced feature set for MLP that replaces high-cardinality categorical features (`ORIGIN`) with continuous graph-based representations (PageRank) to avoid the dimensionality issues caused by one-hot encoding in dense neural networks.

In [0]:
def run_experiment(model, pipeline_stages, train_df, test_df, model_name, dataset_name):
    full_pipeline = Pipeline(stages=pipeline_stages + [model])

    start = time.time()
    fitted = full_pipeline.fit(train_df)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_df)
    test_preds = fitted.transform(test_df)

    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    print(f"\n============================================================")
    print(f"{model_name} on {dataset_name}")
    print(f"Training time: {train_time}s")
    print(f"\n============================================================")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Train Time (s)": train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }, fitted

### Class Imbalance Handling

The training data exhibits a natural class imbalance of approximately 80% on-time flights and 20% delayed flights. Without correction, models trained on this distribution would be biased toward predicting the majority class (on-time), resulting in high accuracy but poor recall for delays.

We apply **random undersampling** to the majority class (on-time flights) to match the minority class count, creating a 50/50 balanced training set. This forces models to learn delay patterns rather than defaulting to the majority class. Undersampling is applied **only to training data**, and all test and validation sets retain their natural class distribution to accurately simulate real-world conditions. During cross-validation, undersampling is performed independently within each fold after the temporal split.

In [0]:
def undersample_majority(df, label_col="DEP_DEL15", seed=42):
    """Undersample the majority class (label=0) to match the minority class count."""
    major_df = df.filter(col(label_col) == 0)
    minor_df = df.filter(col(label_col) == 1)
    major_count = major_df.count()
    minor_count = minor_df.count()
    undersample_ratio = minor_count / major_count
    major_undersampled = major_df.sample(withReplacement=False, fraction=undersample_ratio, seed=seed)
    return major_undersampled.unionAll(minor_df)

   
## Cross-Validation (2015 - 2018)

We use **yearly single-year cross-validation** on the 4-year training set:

| Fold | Train | Validate |
|------|-------|----------|
| 1 | 2015 | 2016 |
| 2 | 2016 | 2017 |
| 3 | 2017 | 2018 |

Each fold trains on one year and validates on the next, providing three independent temporal evaluations. After cross-validation, we train on the full 2015–2018 dataset and evaluate on the held-out 2019 test set.

### Leakage Prevention

To ensure no future information leaks into training, we enforce the following safeguards in every fold:

1. **Temporal separation:** Each fold trains only on a past year and validates on the next year. No random shuffling across time.
2. **Undersampling on train only:** Majority-class downsampling is applied only to the training fold. Validation and test data retain their natural class distribution to accurately simulate real-world conditions.
3. **No target leakage:** We predict `DEP_DEL15` (binary delay indicator) using only features available before departure: weather observations, schedule metadata, and airport/carrier identifiers. No post-departure columns (actual delay minutes, arrival info) are included.

### Overfitting Detection

We report **both train and validation F2** for every fold. A large gap (train >> validation) signals overfitting, the model memorizes training data but fails to generalize, in contrast, smaller gap indicates better generalization.

In [0]:
1

In [0]:
def rolling_cv(df, model_fn, pipeline_stages, folds, label_col="DEP_DEL15"):
    """Rolling cross-validation with yearly folds and majority undersampling.
    Reports both train and test metrics per fold for overfitting analysis."""
    fold_results = []
    fitted_models = []

    for i, fold in enumerate(folds, 1):
        train_fold = df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        # Undersample majority class in the training fold
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        model = model_fn()
        full_pipeline = Pipeline(stages=pipeline_stages + [model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        fitted_models.append(fitted)

        # Evaluate on both train and test
        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    fold_df = pd.DataFrame(fold_results)
    return fold_df, fitted_models

In [0]:
cv_folds = [
    {"train_start": 2015, "train_end": 2015, "test_start": 2016, "test_end": 2016, "label": "Train 2015, Val 2016"},
    {"train_start": 2016, "train_end": 2016, "test_start": 2017, "test_end": 2017, "label": "Train 2016, Val 2017"},
    {"train_start": 2017, "train_end": 2017, "test_start": 2018, "test_end": 2018, "label": "Train 2017, Val 2018"},
]

## Model Comparison with Default Hyperparameters

We run rolling CV on all four models using default hyperparameters to establish baseline performance. Each model is trained on 2 yearly folds with undersampling, reporting train and test metrics per fold to assess both performance and overfitting.

### Logistic Regression (Baseline)

In [0]:
# lr_cv = load_checkpoint("cv3_logistic_regression")
lr_cv = None
if lr_cv is None:
    lr_cv, lr_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        pipeline_stages=lr_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(lr_cv, "cv3_logistic_regression")
else:
    lr_cv_models = []

# Logistic Regression (Baseline) Train/Val Results
Fold 1: Train 2015, Val 2016 (22.9s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6369   0.4935  +0.1434
  Recall         0.6325   0.6070  +0.0255
  Precision      0.6551   0.2823  +0.3728
  AUC-PR         0.7035   0.3472  +0.3563

Fold 2: Train 2016, Val 2017 (21.3s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6362   0.5159  +0.1203
  Recall         0.6317   0.6406  -0.0089
  Precision      0.6550   0.2901  +0.3649
  AUC-PR         0.7039   0.3642  +0.3397

Fold 3: Train 2017, Val 2018 (20.7s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6393   0.5069  +0.1324
  Recall         0.6346   0.6217  +0.0129
  Precision      0.6586   0.2916  +0.3670
  AUC-PR         0.7075   0.3596  +0.3479
Checkpoint saved: 041226_cv3_logistic_regression

### Random Forest

In [0]:
# rf_cv = load_checkpoint("cv3_random_forest")
rf_cv = None
if rf_cv is None:
    rf_cv, rf_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
        pipeline_stages=tree_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(rf_cv, "cv3_random_forest")
else:
    rf_cv_models = []

# Random Forest Train/Val Results
Fold 1: Train 2015, Val 2016 (55.7s)

  Metric          Train     Test      Gap
  --------------------------------------
  
  F2             0.6220   0.4834  +0.1386
  Recall         0.6153   0.5892  +0.0261
  Precision      0.6503   0.2814  +0.3689
  AUC-PR         0.6987   0.3405  +0.3582

Fold 2: Train 2016, Val 2017 (45.8s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6212   0.5067  +0.1145
  Recall         0.6145   0.6291  -0.0146
  Precision      0.6494   0.2850  +0.3644
  AUC-PR         0.7012   0.3616  +0.3396

Fold 3: Train 2017, Val 2018 (44.1s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6152   0.4994  +0.1158
  Recall         0.6043   0.6064  -0.0021
  Precision      0.6628   0.2927  +0.3701
  AUC-PR         0.7073   0.3576  +0.3497
Checkpoint saved: 041226_cv3_random_forest

### Gradient Boosted Trees

In [0]:
# gbt_cv = load_checkpoint("cv3_gradient_boosted_trees")

gbt_cv = None
if gbt_cv is None:
    gbt_cv, gbt_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
        pipeline_stages=tree_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(gbt_cv, "cv3_gradient_boosted_trees")
else:
    gbt_cv_models = []

### Multi Layer Perceptron (With One Hidden Layer)

Feed forward neural network with one hidden layer (`[input_dim, 64, 2]`, `maxIter=100`). Uses `mlp_pipeline_stages` which drops `ORIGIN` one-hot encoding (300+ airports will cause huge sparse vector) and relies on continuous PageRank features instead, because dense neural networks perform poorly with high dimensional sparse inputs.

In [0]:
# mlp_cv = load_checkpoint("cv3_mlp_shallow")
mlp_cv = None
if mlp_cv is None:
    # Custom CV loop: MLP input dimension varies per fold because each temporal
    # split has different categorical cardinalities after OHE.
    fold_results = []
    mlp_cv_models = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        # Determine actual feature dimension for this fold's training data
        prep_pipeline = Pipeline(stages=mlp_pipeline_stages).fit(train_balanced)
        sample_vec = prep_pipeline.transform(train_balanced.limit(1)).select("features").head()[0]
        fold_input_dim = len(sample_vec)

        mlp_model = MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[fold_input_dim, 64, 2], maxIter=100, blockSize=128, seed=42
        )

        full_pipeline = Pipeline(stages=mlp_pipeline_stages + [mlp_model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        mlp_cv_models.append(fitted)

        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    mlp_cv = pd.DataFrame(fold_results)
    save_checkpoint(mlp_cv, "cv3_mlp_shallow")
else:
    mlp_cv_models = []

# MLP (Shallow) Training Results
Fold 1: Train 2015, Val 2016 (86.2s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.5084   0.4107  +0.0977
  Recall         0.4793   0.4501  +0.0292
  Precision      0.6714   0.3043  +0.3671
  AUC-PR         0.6752   0.3264  +0.3488

Fold 2: Train 2016, Val 2017 (97.0s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.5957   0.4923  +0.1034
  Recall         0.5853   0.6068  -0.0215
  Precision      0.6410   0.2806  +0.3604
  AUC-PR         0.6730   0.3374  +0.3356

Fold 3: Train 2017, Val 2018 (101.4s)
  Metric          Train     Test      Gap
  --------------------------------------
  F2             0.6239   0.5054  +0.1185
  Recall         0.6217   0.6620  -0.0403
  Precision      0.6330   0.2597  +0.3733
  AUC-PR         0.6838   0.3420  +0.3418
Checkpoint saved: 041226_cv3_mlp_shallow

In [0]:
# mlp_cv = load_checkpoint("cv_mlp_shallow")
# if mlp_cv is None:
#     mlp_cv, mlp_cv_models = rolling_cv(
#         otpw_train_df,
#         model_fn=lambda: MultilayerPerceptronClassifier(
#             featuresCol="features", labelCol=label_col,
#             layers=[mlp_input_dim, 64, 2], maxIter=100, blockSize=128, seed=42
#         ),
#         pipeline_stages=mlp_pipeline_stages, folds=cv_folds
#     )
#     save_checkpoint(mlp_cv, "cv_mlp_shallow")
# else:
#     mlp_cv_models = []

   
### Cross-Validation Results: Train vs. Validation Comparison

The chart below shows train and validation metrics side-by-side for each model across all three CV folds. The gap values above each bar pair indicate the degree of overfitting: larger positive gaps mean the model is memorizing training data rather than generalizing.

In [0]:
models = ["Logistic Regression", "Random Forest", "Gradient Boosted Trees", "MLP Shallow"]
model_data = {"Logistic Regression": lr_cv, "Random Forest": rf_cv, "Gradient Boosted Trees": gbt_cv, "MLP Shallow": mlp_cv}
metrics = ["F2", "Recall", "Precision", "AUC-PR"]

fig, axes = plt.subplots(len(metrics), len(models), figsize=(6 * len(models), 4 * len(metrics)), sharey="row")

fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(models):
        ax = axes[row, col_idx]
        df = model_data[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Validation", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("Phase III Rolling CV: Train vs Validation (Yearly Folds)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# models = ["Logistic Regression", "Random Forest", "Gradient Boosted Trees", "MLP Shallow"]
# model_data = {"Logistic Regression": lr_cv, "Random Forest": rf_cv, "Gradient Boosted Trees": gbt_cv, "MLP Shallow": mlp_cv}
# metrics = ["F2", "Recall", "Precision", "AUC-PR"]

# fig, axes = plt.subplots(len(metrics), len(models), figsize=(6 * len(models), 4 * len(metrics)), sharey="row")

# fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
# x = np.arange(len(fold_labels))
# width = 0.3

# for row, metric in enumerate(metrics):
#     for col_idx, model_name in enumerate(models):
#         ax = axes[row, col_idx]
#         df = model_data[model_name]
#         train_col = f"Train_{metric}"
#         test_col = f"Test_{metric}"

#         ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
#         ax.bar(x + width/2, df[test_col], width, label="Validation", color="darkorange")
#         ax.set_xticks(x)
#         ax.set_xticklabels(fold_labels, fontsize=8)
#         ax.grid(True, alpha=0.3, axis="y")

#         for j in range(len(df)):
#             gap = df[train_col].iloc[j] - df[test_col].iloc[j]
#             y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
#             ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

#         if col_idx == 0:
#             ax.set_ylabel(metric)
#         if row == 0:
#             ax.set_title(model_name)
#         if row == 0 and col_idx == len(models) - 1:
#             ax.legend(fontsize=8)

# fig.suptitle("Phase III Rolling CV: Train vs Validation (Yearly Folds)", fontsize=14, y=1.01)
# plt.tight_layout()
# plt.show()

   
## Final Evaluation: Train 2015–2018, Test 2019

After cross-validation, we train each model on the **full 2015–2018 training set** (with undersampling) and evaluate on the **held-out 2019 blind test set**. This provides the definitive performance estimate for each model on unseen future data.

In [0]:
# final_results_df = load_checkpoint("cv3_final_results_2019")
final_results_df = None
if final_results_df is None:
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)
    final_results = []
    final_models = {}

    # Logistic Regression
    lr_result, lr_final = run_experiment(
        LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        lr_pipeline_stages, train_balanced_full, otpw_test_df,
        "Logistic Regression", "Train 2015-2018, Test 2019"
    )
    final_results.append(lr_result)
    final_models["lr"] = lr_final

    # Random Forest
    rf_result, rf_final = run_experiment(
        RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
        tree_pipeline_stages, train_balanced_full, otpw_test_df,
        "Random Forest", "Train 2015-2018, Test 2019"
    )
    final_results.append(rf_result)
    final_models["rf"] = rf_final

    # Gradient Boosted Trees
    gbt_result, gbt_final = run_experiment(
        GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
        tree_pipeline_stages, train_balanced_full, otpw_test_df,
        "Gradient Boosted Trees", "Train 2015-2018, Test 2019"
    )
    final_results.append(gbt_result)
    final_models["gbt"] = gbt_final

    # MLP (mlp_input_dim is valid here since it was computed from full 2015-2018 data)
    mlp_result, mlp_final = run_experiment(
        MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[mlp_input_dim, 64, 2], maxIter=100, blockSize=128, seed=42
        ),
        mlp_pipeline_stages, train_balanced_full, otpw_test_df,
        "MLP Shallow", "Train 2015-2018, Test 2019"
    )
    final_results.append(mlp_result)
    final_models["mlp"] = mlp_final

    final_results_df = pd.DataFrame(final_results)
    save_checkpoint(final_results_df, "cv3_final_results_2019")
else:
    final_models = {}

display(final_results_df)

============================================================
Logistic Regression on Train 2015-2018, Test 2019
Training time: 33.5s

============================================================
Metric                    Train       Test
------------------------------------------
AUC-PR                   0.7008     0.3551
Precision                0.6530     0.2932
Recall                   0.6280     0.6133
F2                       0.6328     0.5034

============================================================
Random Forest on Train 2015-2018, Test 2019
Training time: 123.3s

============================================================
Metric                    Train       Test
------------------------------------------
AUC-PR                   0.6953     0.3463
Precision                0.6527     0.2892
Recall                   0.6067     0.6035
F2                       0.6154     0.4957

============================================================
Gradient Boosted Trees on Train 2015-2018, Test 2019
Training time: 486.9s

============================================================
Metric                    Train       Test
------------------------------------------
AUC-PR                   0.7165     0.3801
Precision                0.6676     0.3035
Recall                   0.6147     0.5990
F2                       0.6246     0.5014

============================================================
MLP Shallow on Train 2015-2018, Test 2019
Training time: 240.7s

============================================================
Metric                    Train       Test
------------------------------------------
AUC-PR                   0.6754     0.3411
Precision                0.6388     0.2706
Recall                   0.5678     0.6040
F2                       0.5807     0.4846
Checkpoint saved: 041226_cv3_final_results_2019

In [0]:
final_results_df.to_html()

# HTML Table of Final Results DF

<table border="1" class="dataframe">  <thead>    <tr style="text-align: right;">      <th></th>      <th>Model</th>      <th>Dataset</th>      <th>Train Time (s)</th>      <th>Train_AUC-PR</th>      <th>Train_Precision</th>      <th>Train_Recall</th>      <th>Train_F2</th>      <th>Test_AUC-PR</th>      <th>Test_Precision</th>      <th>Test_Recall</th>      <th>Test_F2</th>    </tr>  </thead>  <tbody>    <tr>      <th>0</th>      <td>Logistic Regression</td>      <td>Train 2015-2018, Test 2019</td>      <td>33.5</td>      <td>0.7008</td>      <td>0.6530</td>      <td>0.6280</td>      <td>0.6328</td>      <td>0.3551</td>      <td>0.2932</td>      <td>0.6133</td>      <td>0.5034</td>    </tr>    <tr>      <th>1</th>      <td>Random Forest</td>      <td>Train 2015-2018, Test 2019</td>      <td>123.3</td>      <td>0.6953</td>      <td>0.6527</td>      <td>0.6067</td>      <td>0.6154</td>      <td>0.3463</td>      <td>0.2892</td>      <td>0.6035</td>      <td>0.4957</td>    </tr>    <tr>      <th>2</th>      <td>Gradient Boosted Trees</td>      <td>Train 2015-2018, Test 2019</td>      <td>486.9</td>      <td>0.7165</td>      <td>0.6676</td>      <td>0.6147</td>      <td>0.6246</td>      <td>0.3801</td>      <td>0.3035</td>      <td>0.5990</td>      <td>0.5014</td>    </tr>    <tr>      <th>3</th>      <td>MLP Shallow</td>      <td>Train 2015-2018, Test 2019</td>      <td>240.7</td>      <td>0.6754</td>      <td>0.6388</td>      <td>0.5678</td>      <td>0.5807</td>      <td>0.3411</td>      <td>0.2706</td>      <td>0.6040</td>      <td>0.4846</td>    </tr>  </tbody></table>